# DSPy — Otimização com `BootstrapFewShot`

Neste notebook será demonstrado o uso do otimizador `BootstrapFewShotWithRandomSearch` do DSPy em um problema de **classificação binária de textos**.

Será utilizada a base **Natural Language Processing with Disaster Tweets**, disponibilizada no Kaggle. O objetivo é classificar cada tweet em uma das seguintes categorias:

* `0`: o tweet **não descreve um desastre real**;
* `1`: o tweet **descreve um desastre real**.

O experimento será dividido em duas etapas:

1. avaliar um classificador DSPy sem exemplos de demonstração (**zero-shot**);
2. otimizar o classificador utilizando `BootstrapFewShotWithRandomSearch` e avaliá-lo novamente.

A métrica principal utilizada para comparar os dois programas será o **F1-score**.

In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env
import dspy  # Framework para otimização de prompts com Language Models

import pandas as pd

from typing import Literal
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)

from tqdm.auto import tqdm

/home/leonardo/Documentos/github/dspy_studies/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()  # Lê variáveis do arquivo .env

True

## Setup - Configuração do Modelo

Primeiro, carregamos as variáveis de ambiente (como API key) do arquivo `.env` na raiz do projeto. Isso evita hardcoding de credenciais no código.

In [3]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Modelo OpenAI GPT-5 Mini (modelo compacto e rápido)
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada da variável de ambiente
)

# Configura o modelo padrão para todas as operações DSPy
dspy.configure(lm=lm)

## 3. Leitura da base de dados

Será utilizada a base da competição **Natural Language Processing with Disaster Tweets**, do Kaggle.

Referência:

https://www.kaggle.com/competitions/nlp-getting-started

Para este experimento são relevantes principalmente duas colunas:

| Coluna   | Descrição                   |
| -------- | --------------------------- |
| `text`   | Texto do tweet              |
| `target` | Classe correta (`0` ou `1`) |

O problema consiste, portanto, em aprender a relação:

`text → target`


In [4]:
df = pd.read_csv('disaster_tweets.csv')
df = df.head(200)
df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


## Análise da distribuição das classes

Antes da divisão dos dados, é importante verificar quantos exemplos existem de cada classe.

Além da quantidade absoluta, será analisada a proporção entre tweets classificados como `0` e `1`.

Essa análise é importante porque o **F1-score** considera conjuntamente precisão e recall e é especialmente útil quando existe algum grau de desbalanceamento entre as classes.


In [5]:
df["target"].value_counts()

target
0    103
1     97
Name: count, dtype: int64

In [6]:
df["target"].value_counts(normalize=True)

target
0    0.515
1    0.485
Name: proportion, dtype: float64

## Separação entre treino, validação e teste

A base será dividida em:

* **64% para treinamento**: criação das demonstrações
* **16% para validação**: escolha do melhor programa candidato
* **20% para teste**: avaliação final

O parâmetro `stratify=df["target"]` é utilizado para manter aproximadamente a mesma proporção entre as classes `0` e `1` nos dois conjuntos.

Essa separação evita que exemplos utilizados durante a otimização sejam empregados também na avaliação final.


In [7]:
df_train_val, df_test = train_test_split(
    df[["text", "target"]],
    test_size=0.20,
    random_state=42,
    stratify=df["target"],
)

df_train, df_val = train_test_split(
    df_train_val,
    test_size=0.20,
    random_state=42,
    stratify=df_train_val["target"],
)

print(f"Treino:     {len(df_train)} exemplos")
print(f"Validação:  {len(df_val)} exemplos")
print(f"Teste:      {len(df_test)} exemplos")

Treino:     128 exemplos
Validação:  32 exemplos
Teste:      40 exemplos


## Conversão para `dspy.Example`

O DSPy representa exemplos de treino e teste por meio da classe `dspy.Example`.

Neste problema, cada exemplo possui dois campos:

* `text`: entrada fornecida ao modelo;
* `target`: resposta esperada.

A chamada:

`with_inputs("text")`

informa explicitamente ao DSPy que `text` deve ser utilizado como entrada do programa.

Consequentemente, `target` passa a ser tratado como o **label**, ou seja, a resposta esperada para aquele exemplo.

Conceitualmente, cada registro passa a ter a seguinte estrutura:

`entrada: text → saída esperada: target`

In [8]:
def dataframe_para_dspy(dataframe):
    exemplos = []

    for _, row in dataframe.iterrows():
        exemplo = dspy.Example(
            text=row["text"],
            target=int(row["target"]),
        ).with_inputs("text")

        exemplos.append(exemplo)

    return exemplos

In [9]:
trainset = dataframe_para_dspy(df_train)
valset = dataframe_para_dspy(df_val)
testset = dataframe_para_dspy(df_test)

print(f"Trainset DSPy: {len(trainset)}")
print(f"Valset DSPy:   {len(valset)}")
print(f"Testset DSPy:  {len(testset)}")

Trainset DSPy: 128
Valset DSPy:   32
Testset DSPy:  40


In [10]:
trainset[0]

Example({'text': 'AMBULANCE SPRINTER AUTOMATIC FRONTLINE VEHICLE CHOICE OF 14 LEZ COMPLIANT | eBay http://t.co/UJrX9kgawp', 'target': 0}) (input_keys={'text'})

## Definição da tarefa com uma `Signature`

No DSPy, uma `Signature` descreve declarativamente a tarefa que será executada pelo modelo.

A `ClassificarTweet` possui:

* um `InputField` chamado `text`, contendo o tweet;
* um `OutputField` chamado `target`, contendo a classificação.

O tipo:

`Literal[0, 1]`

restringe a resposta esperada às duas classes válidas do problema.

Dessa forma, a Signature define claramente o contrato:

`texto do tweet → 0 ou 1`


In [11]:
class ClassificarTweet(dspy.Signature):
    """
    Determine se o tweet descreve um desastre real.

    Retorne:
    - 1 se o tweet estiver relacionado a um desastre real.
    - 0 caso contrário.
    """

    text: str = dspy.InputField(
        desc="Texto do tweet que deve ser classificado."
    )

    target: Literal[0, 1] = dspy.OutputField(
        desc="1 para desastre real e 0 para não desastre."
    )

In [12]:
classificador_base = dspy.Predict(ClassificarTweet)

In [13]:
def avaliar_classificador(programa, dataset, descricao="Avaliando"):
    """
    Executa um programa DSPy sobre um dataset e calcula
    métricas globais de classificação.
    """

    y_true = []
    y_pred = []

    for exemplo in tqdm(dataset, desc=descricao):

        # Executa o programa utilizando somente os campos
        # marcados como entrada pelo with_inputs(...)
        predicao = programa(**exemplo.inputs())

        # Label verdadeiro
        y_true.append(int(exemplo.target))

        # Label previsto pelo DSPy
        y_pred.append(int(predicao.target))

    # Calcula as métricas sobre todo o conjunto
    resultado = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

    return resultado

In [14]:
resultado_base = avaliar_classificador(
    classificador_base,
    testset,
    descricao="Baseline zero-shot",
)

Baseline zero-shot: 100%|███████████████████████████████████| 40/40 [00:02<00:00, 19.72it/s]


## Avaliação do baseline

O classificador inicial será executado sobre todos os exemplos do conjunto de teste.

Para cada exemplo:

1. o campo `text` é enviado ao programa;
2. o programa produz uma previsão para `target`;
3. a previsão é comparada com o `target` verdadeiro.

Ao final são calculadas as métricas globais de classificação.

Esse resultado será considerado o desempenho **antes da otimização**.


In [15]:
print(f"F1 baseline: {resultado_base['f1']:.4f}")

F1 baseline: 0.9474


In [16]:
print(f"Accuracy:  {resultado_base['accuracy']:.4f}")
print(f"Precision: {resultado_base['precision']:.4f}")
print(f"Recall:    {resultado_base['recall']:.4f}")
print(f"F1:        {resultado_base['f1']:.4f}")

Accuracy:  0.9500
Precision: 0.9474
Recall:    0.9474
F1:        0.9474


## Otimização com `BootstrapFewShotWithRandomSearch`

O `BootstrapFewShotWithRandomSearch` é um otimizador do DSPy que estende o funcionamento do `BootstrapFewShot`.

Enquanto o `BootstrapFewShot` constrói um único programa otimizado a partir de demonstrações rotuladas e demonstrações geradas por bootstrap, o `BootstrapFewShotWithRandomSearch` cria **vários programas candidatos**, utilizando diferentes combinações de demonstrações.

Esses candidatos são avaliados em um conjunto de validação (`valset`), e o programa que obtiver o melhor resultado é selecionado como programa otimizado.

Neste experimento será utilizado:

```python
dspy.BootstrapFewShotWithRandomSearch(
    metric=metrica_bootstrap,
    max_bootstrapped_demos=4,
    max_labeled_demos=16,
    max_rounds=1,
    num_candidate_programs=8,
)
```

A otimização será realizada utilizando:

```python
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)
```

Neste caso, os conjuntos possuem funções diferentes:

* `trainset`: utilizado para construir as demonstrações dos programas candidatos;
* `valset`: utilizado para avaliar os programas candidatos e selecionar o melhor;
* `testset`: utilizado somente após a otimização para medir o desempenho final do classificador.

Essa separação é importante porque o conjunto de teste não deve participar da escolha do programa otimizado.

---

### Métrica utilizada durante a otimização

Neste experimento será utilizada uma métrica de acerto exato:

```python
def metrica_bootstrap(example, prediction, trace=None):
    return int(example.target) == int(prediction.target)
```

Essa métrica verifica, para cada exemplo individualmente, se a classe prevista corresponde à classe esperada.

No `BootstrapFewShotWithRandomSearch`, essa métrica possui dois papéis principais:

1. verificar quais execuções geradas durante o bootstrap podem ser utilizadas como demonstrações;
2. avaliar o desempenho dos programas candidatos no `valset`.

Como a métrica retorna `True` ou `False` para cada exemplo, o score agregado utilizado para comparar os candidatos corresponde, neste caso, ao percentual de exemplos classificados corretamente.

É importante diferenciar esse score da métrica F1 utilizada na avaliação final:

* `metrica_bootstrap`: utilizada internamente durante a otimização e seleção dos candidatos;
* `F1`: utilizada posteriormente para avaliar o programa selecionado sobre todo o `testset`.

---

### Parâmetros utilizados

O parâmetro:

```python
max_bootstrapped_demos=4
```

define o número máximo de demonstrações produzidas pelo processo de bootstrap que podem ser utilizadas em cada programa candidato.

Essas demonstrações são obtidas a partir das execuções de um **teacher** e somente são aceitas quando satisfazem a métrica definida.

---

O parâmetro:

```python
max_labeled_demos=16
```

define o número máximo de demonstrações rotuladas diretamente provenientes do `trainset` que podem ser incluídas no programa.

Portanto, as demonstrações utilizadas por um programa podem combinar:

```text
demonstrações geradas por bootstrap
+
demonstrações rotuladas do trainset
```

---

O parâmetro:

```python
max_rounds=1
```

define o número máximo de tentativas de bootstrap realizadas para cada exemplo de treinamento.

Caso sejam utilizadas várias rodadas, o DSPy pode realizar novas execuções para tentar obter uma demonstração que satisfaça a métrica.

---

O parâmetro:

```python
num_candidate_programs=8
```

controla quantos conjuntos de candidatos baseados em busca aleatória serão explorados.

Para gerar esses candidatos, o DSPy modifica fatores como:

* a ordem dos exemplos do `trainset`;
* a quantidade de demonstrações geradas por bootstrap;
* a combinação de demonstrações utilizada pelo programa.

Além dos candidatos produzidos pela busca aleatória, o otimizador também considera configurações especiais, como:

```text
zero-shot
labels-only
BootstrapFewShot sem embaralhamento
```

Isso permite comparar diferentes estratégias antes de selecionar o programa vencedor.

---

### Funcionamento da busca

Conceitualmente, o processo pode ser representado da seguinte forma:

```text
                         trainset
                            │
                            ↓
             geração de diferentes candidatos
                            │
             ┌──────────────┼──────────────┐
             ↓              ↓              ↓
        candidato 1    candidato 2    candidato N
             │              │              │
             └──────────────┼──────────────┘
                            ↓
                          valset
                            │
                            ↓
                 métrica avalia cada
                      candidato
                            │
                            ↓
                   comparação dos scores
                            │
                            ↓
                   melhor programa
                            │
                            ↓
              classificador_otimizado
```

---

### Antes da otimização

Antes da otimização, o classificador opera sem demonstrações selecionadas pelo otimizador:

```text
instrução
    +
novo tweet
    ↓
modelo
    ↓
classificação
```

---

### Durante a otimização

Durante a otimização:

```text
trainset
   ↓
gera diferentes conjuntos de demonstrações
   ↓
constrói vários programas candidatos
   ↓
executa cada candidato no valset
   ↓
métrica calcula o score de cada candidato
   ↓
compara os candidatos
   ↓
seleciona o programa com melhor score
```

---

### Depois da otimização

O programa selecionado contém as demonstrações pertencentes ao melhor candidato encontrado:

```text
instrução
    +
demonstrações do melhor candidato
    +
novo tweet
    ↓
modelo
    ↓
classificação
```

O programa selecionado é então avaliado separadamente utilizando o `testset`:

```text
classificador_otimizado
          +
       testset
          ↓
Accuracy
Precision
Recall
F1
```

---

Portanto, o `BootstrapFewShotWithRandomSearch` **não altera os pesos do modelo de linguagem**.

A otimização ocorre através da busca por uma configuração de demonstrações que produza o melhor desempenho no conjunto de validação.

A principal diferença em relação ao `BootstrapFewShot` é que, em vez de construir apenas uma configuração de demonstrações, o `BootstrapFewShotWithRandomSearch` explora **múltiplos programas candidatos**, avalia cada um deles e seleciona automaticamente o melhor.


In [17]:
def metrica_bootstrap(example, prediction, trace=None):
    """
    Verifica se a classificação prevista é exatamente igual
    à classificação esperada.

    Essa métrica é utilizada pelo BootstrapFewShot para decidir
    se uma execução pode ser utilizada como demonstração.
    """

    return int(example.target) == int(prediction.target)

In [18]:
optimizer = dspy.BootstrapFewShotWithRandomSearch(
    metric=metrica_bootstrap,
    max_bootstrapped_demos=4,
    max_labeled_demos=16,
    max_rounds=1,
    num_candidate_programs=8,
)

Going to sample between 1 and 4 traces per predictor.
Will attempt to bootstrap 8 candidate sets.


## Compilação do programa otimizado

O método `compile()` recebe:

* o programa original (`student`);
* o conjunto de treinamento (`trainset`);

In [19]:
classificador_otimizado = optimizer.compile(
    student=classificador_base,
    trainset=trainset,
    valset=valset,
)

Average Metric: 30.00 / 32 (93.8%): 100%|███████████████████| 32/32 [00:42<00:00,  1.34s/it]

2026/09/07 18:19:49 INFO dspy.evaluate.evaluate: Average Metric: 30 / 32 (93.8%)



New best score: 93.75 for seed -3
Scores so far: [93.75]
Best score so far: 93.75
Average Metric: 30.00 / 32 (93.8%): 100%|███████████████████| 32/32 [00:25<00:00,  1.27it/s]

2026/09/07 18:20:14 INFO dspy.evaluate.evaluate: Average Metric: 30 / 32 (93.8%)



Scores so far: [93.75, 93.75]
Best score so far: 93.75


...


Scores so far: [93.75, 93.75, 96.88, 87.5, 93.75, 90.62, 93.75, 90.62, 93.75, 90.62]
Best score so far: 96.88


  2%|█▎                                                     | 3/128 [00:17<12:26,  5.98s/it]


Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Average Metric: 30.00 / 32 (93.8%): 100%|███████████████████| 32/32 [00:27<00:00,  1.16it/s]

2026/09/07 18:27:02 INFO dspy.evaluate.evaluate: Average Metric: 30 / 32 (93.8%)



Scores so far: [93.75, 93.75, 96.88, 87.5, 93.75, 90.62, 93.75, 90.62, 93.75, 90.62, 93.75]
Best score so far: 96.88
11 candidate programs found.


## Inspeção das demonstrações selecionadas

Depois da compilação, as demonstrações adicionadas ao programa podem ser acessadas pela propriedade:

`demos`

A inspeção desses exemplos permite visualizar concretamente como o `BootstrapFewShotWithRandomSearch` modificou o programa.

Cada demonstração contém:

* o texto de entrada (`text`);
* a classificação correta (`target`).

Esses exemplos passam a fazer parte do contexto fornecido ao modelo quando uma nova classificação é realizada.


In [20]:
demos = classificador_otimizado.demos

print(f"Número de demonstrações: {len(demos)}")

Número de demonstrações: 16


In [21]:
for i, exemplo in enumerate(demos, start=1):
    print(f"--- Exemplo {i} ---")
    print(f"Tweet:  {exemplo.text}")
    print(f"Target: {exemplo.target}")
    print()

--- Exemplo 1 ---
Tweet:  AMBULANCE SPRINTER AUTOMATIC FRONTLINE VEHICLE CHOICE OF 14 LEZ COMPLIANT | eBay http://t.co/UJrX9kgawp
Target: 0

--- Exemplo 2 ---
Tweet:  Statistically I'm at more of risk of getting killed by a cop than I am of dying in an airplane accident.
Target: 0

--- Exemplo 3 ---
Tweet:  @ablaze what time does your talk go until? I don't know if I can make it due to work.
Target: 0

--- Exemplo 4 ---
Tweet:  AMBULANCE SPRINTER AUTOMATIC FRONTLINE VEHICLE CHOICE OF 14 LEZ COMPLIANT | eBay http://t.co/Kp2Lf4AuTe
Target: 0

--- Exemplo 5 ---
Tweet:  on the outside you're ablaze and alive
but you're dead inside
Target: 0

--- Exemplo 6 ---
Tweet:  @AlexAllTimeLow awwww they're on an airplane accident and they're gonna die what a cuties ???? good job!
Target: 1

--- Exemplo 7 ---
Tweet:  #stlouis #caraccidentlawyer Speeding Among Top Causes of Teen Accidents https://t.co/k4zoMOF319 https://t.co/S2kXVM0cBA Car Accident teeÛ_
Target: 0

--- Exemplo 8 ---
Tweet:  OMG Horri

In [22]:
resultado_otimizado = avaliar_classificador(
    classificador_otimizado,
    testset,
    descricao="BootstrapFewShotWithRandomSearch",
)

BootstrapFewShotWithRandomSearch: 100%|█████████████████████| 40/40 [04:13<00:00,  6.34s/it]


In [23]:
print(f"F1 otimizado: {resultado_otimizado['f1']:.4f}")

F1 otimizado: 0.9730


## Avaliação após a otimização

O programa otimizado será avaliado utilizando **exatamente o mesmo conjunto de teste utilizado pelo baseline**.

Isso permite uma comparação justa entre:

* classificador original;
* classificador com `BootstrapFewShotWithRandomSearch`.

Nenhum exemplo do conjunto de teste participa da seleção das demonstrações.

Ao final, será novamente calculado o F1-score juntamente com Accuracy, Precision e Recall.


In [24]:
comparacao = pd.DataFrame(
    {
        "Modelo": [
            "Baseline (zero-shot)",
            "BootstrapFewShot",
        ],
        "Accuracy": [
            resultado_base["accuracy"],
            resultado_otimizado["accuracy"],
        ],
        "Precision": [
            resultado_base["precision"],
            resultado_otimizado["precision"],
        ],
        "Recall": [
            resultado_base["recall"],
            resultado_otimizado["recall"],
        ],
        "F1": [
            resultado_base["f1"],
            resultado_otimizado["f1"],
        ],
    }
)

comparacao

,Modelo,Accuracy,Precision,Recall,F1
0,Baseline (zero-shot),0.950,0.947368,0.947368,0.947368
1,BootstrapFewShot,0.975,1.000000,0.947368,0.972973


In [25]:
print("BASELINE")
print(
    classification_report(
        resultado_base["y_true"],
        resultado_base["y_pred"],
        digits=4,
    )
)

BASELINE
              precision    recall  f1-score   support

           0     0.9524    0.9524    0.9524        21
           1     0.9474    0.9474    0.9474        19

    accuracy                         0.9500        40
   macro avg     0.9499    0.9499    0.9499        40
weighted avg     0.9500    0.9500    0.9500        40



In [26]:
print("BootstrapFewShotWithRandomSearch")
print(
    classification_report(
        resultado_otimizado["y_true"],
        resultado_otimizado["y_pred"],
        digits=4,
    )
)

BootstrapFewShotWithRandomSearch
              precision    recall  f1-score   support

           0     0.9545    1.0000    0.9767        21
           1     1.0000    0.9474    0.9730        19

    accuracy                         0.9750        40
   macro avg     0.9773    0.9737    0.9749        40
weighted avg     0.9761    0.9750    0.9750        40



## Persistência do programa otimizado

Após a otimização, o estado do programa será salvo em:

`BootstrapFewShotWithRandomSearch.json`

Ao salvar um módulo DSPy dessa maneira, o arquivo JSON armazena o **estado do programa**, incluindo as demonstrações adicionadas durante a otimização.

Isso permite reutilizar o resultado posteriormente sem precisar executar novamente o `BootstrapFewShotWithRandomSearch`.


In [27]:
classificador_otimizado.save("BootstrapFewShotWithRandomSearch.json")

In [28]:
# Recria a mesma arquitetura do programa
classificador_carregado = dspy.Predict(ClassificarTweet)

# Carrega o estado otimizado salvo pelo LabeledFewShot
classificador_carregado.load("BootstrapFewShotWithRandomSearch.json")
classificador_carregado

Predict(StringSignature(text -> target
    instructions='Determine se o tweet descreve um desastre real.\n\nRetorne:\n- 1 se o tweet estiver relacionado a um desastre real.\n- 0 caso contrário.'
    text = Field(annotation=str required=True json_schema_extra={'desc': 'Texto do tweet que deve ser classificado.', '__dspy_field_type': 'input', 'prefix': 'Text:'})
    target = Field(annotation=Literal[0, 1] required=True json_schema_extra={'desc': '1 para desastre real e 0 para não desastre.', '__dspy_field_type': 'output', 'prefix': 'Target:'})
))

In [29]:
predicao = classificador_carregado(
    text="A massive wildfire is spreading through the forest."
)

print(predicao)

Prediction(
    target=1
)


In [30]:
len(classificador_carregado.demos)

16

In [31]:
# Mostra última chamada ao modelo (n=1 significa 1 última chamada)
dspy.inspect_history(n=1)





[2026-09-07T18:31:38.841976]

System message:

Your input fields are:
1. `text` (str): Texto do tweet que deve ser classificado.
Your output fields are:
1. `target` (Literal[0, 1]): 1 para desastre real e 0 para não desastre.
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "target": "{target}        # note: the value you produce must exactly match (no extra characters) one of: 0; 1"
}
In adhering to this structure, your objective is: 
        Determine se o tweet descreve um desastre real.
        
        Retorne:
        - 1 se o tweet estiver relacionado a um desastre real.
        - 0 caso contrário.


User message:

[[ ## text ## ]]
AMBULANCE SPRINTER AUTOMATIC FRONTLINE VEHICLE CHOICE OF 14 LEZ COMPLIANT | eBay http://t.co/UJrX9kgawp


Assistant message:

{
  "target": 0
}


User message:

[[ ## tex